In [1]:
!pip install -q transformers torch pillow pandas tqdm --quiet

In [2]:
from google.colab import drive
import json
import os

# Mount Drive
drive.mount('/content/drive', force_remount=True)

# Verify paths exist
json_path = "/content/drive/MyDrive/EvaltheEvaluators/ChartGemma_Model/ChartGemma_Descriptions.json"
images_dir = "/content/drive/MyDrive/EvaltheEvaluators/images"

# Check JSON
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    print(f"JSON loaded: {len(data)} samples")
    print(f"Sample: {data[0]}")
else:
    print(f"JSON not found: {json_path}")

# Check images
if os.path.exists(images_dir):
    images = os.listdir(images_dir)
    print(f"Images directory: {len(images)} files")
    print(f"Sample: {images}")
else:
    print(f"Images directory not found: {images_dir}")

Mounted at /content/drive
JSON loaded: 21 samples
Sample: {'img_id': 3128, 'answer': 'The chart shows the infant mortality rate in Armenia from 2009 to 2019. The rate is measured in deaths per 1,000 live births. The chart shows that the rate has been steadily decreasing over the past decade. In 2009, the rate was around 15 deaths per 1,000 live births. By 2019, the rate had fallen to around 10 deaths per 1,000 live births. This represents a significant improvement in the health of infants in Armenia.'}
Images directory: 21 files
Sample: ['3128.png', '2114.png', '2472.png', '8673.png', '1356.png', '1061.png', '2121.png', '8038.png', '4915.png', '7567.png', '2369.png', '623.png', '5932.png', '4067.png', '2264.png', '7127.png', '1064.png', '749.png', '1253.png', '3190.png', '4845.png']


# Load ChartJudge-2B

In [3]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

model_name = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully")
print(f"Model dtype: {next(model.parameters()).dtype}")

# Check memory usage
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f}GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}GB")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully
Model dtype: torch.float16
GPU Memory: 4.42GB / 15.64GB


# Pointwise

In [4]:
!pip install --upgrade transformers --quiet

In [5]:
# Run the evaluation script
exec(open('/content/drive/MyDrive/EvaltheEvaluators/evaluations/ChartJudge_Suite/chartjudge_eval_colab.py').read())

Loading ChartJudge-2B on cuda...


OSError: chartjudge/ChartJudge-2B is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
import os
import pandas as pd
import json

results_dir = "/content/drive/MyDrive/EvaltheEvaluators/ChartJudgeSuite/"

# List output files
print("Generated files:")
for file in os.listdir(results_dir):
    print(f"  - {file}")

# Show pointwise summary
pointwise_csv = os.path.join(results_dir, "pointwise_summary.csv")
if os.path.exists(pointwise_csv):
    df = pd.read_csv(pointwise_csv)
    print(f"\nPointwise Summary ({len(df)} samples):")
    print(df.head())
    print("\nScore Statistics:")
    print(df.describe())

# Show pairwise summary
pairwise_csv = os.path.join(results_dir, "pairwise_summary.csv")
if os.path.exists(pairwise_csv):
    df = pd.read_csv(pairwise_csv)
    print(f"\nPairwise Summary ({len(df)} pairs):")
    print(df.head())

# Show sample detailed results
pointwise_json = os.path.join(results_dir, "pointwise_results.json")
if os.path.exists(pointwise_json):
    with open(pointwise_json, 'r') as f:
        data = json.load(f)
    print(f"\nSample Detailed Result (first sample):")
    print(json.dumps(data[0], indent=2))